In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

from Model.Constants import db, model_db
from Model.DBTypes import *
from Model.ModelDBTypes import *
from Model.DB_AdvancedStatements import *

In [ ]:
cursor = db.cursor()

pitcher_set = {0}
pitcher_first_months : list[tuple[DB_Model_PitcherStats, DB_PitcherStatcastMonth]] = []

years = [2021, 2022, 2023]
months = [4,5,6,7,8,9]

for year in years:
    for month in months:
        low_a_pitchers = Select_InnerJoin(DB_Model_PitcherStats, DB_PitcherStatcastMonth, cursor,
            '''SELECT mps.*, psm.*
            FROM Model_PitcherStats AS mps
            INNER JOIN PitcherStatcastMonth AS psm
                ON mps.mlbId=psm.mlbId
                AND mps.Year=psm.Year
                AND mps.Month=psm.Month
            WHERE mps.Year=?
            AND mps.Month=?
            AND mps.LevelId>=5
            AND mps.BF>30
            ''',
            (year, month)
        )
        
        for p in low_a_pitchers:
            mps, psm = p
            if not mps.mlbId in pitcher_set:
                pitcher_set.add(mps.mlbId)
                pitcher_first_months.append(p)

print(len(pitcher_first_months))

In [ ]:
model_cursor = model_db.cursor()

data : list[tuple[float, float]] = []

for mps, psm in pitcher_first_months:
    try:
        expected_war = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
            "WHERE mlbId=? AND model=1 AND isHitter=0 AND Year=? AND Month=?",
            (mps.mlbId, mps.Year, mps.Month))[0].war
        
        try:
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=0 AND Year=? AND Month=?",
                (mps.mlbId, mps.Year + 2, mps.Month))[0].war
        except:
            # Do last value
            expected_war_2years = DB_Output_PlayerWarAggregation.Select_From_DB(model_cursor,
                "WHERE mlbId=? AND model=1 AND isHitter=0 ORDER BY Year DESC, Month DESC",
                (mps.mlbId,))[0].war
            
        val = psm.StuffFastball
        if expected_war < 3 and val is not None:
            data.append((val, expected_war_2years - expected_war))
    except:
        pass # Not a prospect
    


In [ ]:
def sortFunction(d : tuple[float, float]):
    x, y = d
    return x

data.sort(key=sortFunction)

In [ ]:
l = len(data)
NUM_BINS = 10

quantile_cutoffs = [data[int(l * x / NUM_BINS)][0] for x in range(1, NUM_BINS)]

In [ ]:
bins = [[] for _ in range(NUM_BINS)]

def avg(xs : list[float]) -> float:
    if len(xs) == 0:
        return None
    
    return sum(xs) / len(xs)

for val, delta in data:
    if val > quantile_cutoffs[-1]:
        bins[-1].append(delta)
    else:
        for i, quant_val in enumerate(quantile_cutoffs):
            if val < quant_val:
                bins[i].append(delta)
                break
        
bin_ys = [avg(bin) for bin in bins]
for bin in bin_ys:
    print(f"{bin:.3f}")

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

xs = [d[0] for d in data]
ys = [d[1] for d in data]

x = np.array(xs).reshape(-1, 1)
y = np.array(ys).reshape(-1, 1)

lr = LinearRegression()
lr.fit(x, y)
print(f"Slope (coefficient): {lr.coef_[0][0]:.4f}")
print(f"Intercept: {lr.intercept_[0]:.4f}")
print(f"R² Score: {lr.score(x, y):.4f}")

In [ ]:
min_x = data[0][0]
max_x = data[-1][0]

In [ ]:
neutral_xs = [min_x, max_x]
neutral_ys = [0, 0]

In [ ]:
bin_xs = [(min_x + quantile_cutoffs[0]) / 2] + [(quantile_cutoffs[i] + quantile_cutoffs[i+1]) / 2 for i in range(len(quantile_cutoffs) - 1)] + [(max_x + quantile_cutoffs[-1]) / 2]

In [ ]:
import matplotlib.pyplot as plt

plt.plot(xs, ys, 'ko')
plt.plot(bin_xs, bin_ys, 'r-')
plt.plot(neutral_xs, neutral_ys, 'g--')